# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading, exploration, and processing of the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata and initialize Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata['name'])
print("Description:", metadata['description'])

# Show basic metadata fields
print("Publication Date:", metadata['datePublished'])
print("Identifier:", metadata['identifier'])
print("Keywords:", metadata['keywords'])
print("License:", metadata['license'])

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We display all record sets and their top-level details. If the schema provides record sets, each is referenced by its unique `@id`. Fields (columns) within each record set are also referenced by `@id`.

In [ ]:
# Get all available record sets in the dataset
record_sets = dataset.metadata.record_sets

print(f"Found {len(record_sets)} record sets:\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print(f"  Number of Fields: {len(rs.fields)}")
    print("  Field @ids:")
    for f in rs.fields:
        print(f"    - {f.id}: {f.name}")
    print("")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

We use the `@id` from the overview for each record set and field.

In [ ]:
# Prepare extraction of all record sets
dataframes = {}

for rs in record_sets:
    # Reference by @id
    rs_id = rs.id
    print(f"Extracting records from RecordSet @id={rs_id}, Name={rs.name}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns (@id): {df.columns.tolist()}")
    print(f"Sample data (first 2 rows):\n{df.head(2)}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records by criteria, normalizing numeric fields, grouping by key attributes.

**Note:** All fields are referenced by their `@id`.

For demonstration, we'll select the first record set (typically Table 1: Clinical features), and process a numeric field (e.g., age at diagnosis). We'll also group by a categorical field if available.

In [ ]:
# Choose the first record set for EDA
if record_sets:
    record_set = record_sets[0]
    record_set_id = record_set.id
    df = dataframes[record_set_id]

    print(f"Working with record set @id={record_set_id} ({record_set.name})")

    # Identify a numeric field (e.g., age at diagnosis)
    numeric_field = None
    for field in record_set.fields:
        if getattr(field, 'data_type', '').lower() in ['integer', 'float', 'number'] or 'age' in field.name.lower():
            numeric_field = field.id
            break
    
    # If not found, default to first field
    if not numeric_field:
        numeric_field = record_set.fields[0].id

    print(f"Using numeric field: {numeric_field}")
    
    # Filtering and normalization
    threshold = 20  # Example threshold for age
    if numeric_field in df.columns:
        # Attempt conversion to numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Add normalized column
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field, e.g., 'infertility' ('@id' contains this)
        group_field = None
        for field in record_set.fields:
            if 'infertility' in field.name.lower() or getattr(field, 'data_type', '').lower() == 'boolean':
                group_field = field.id
                break

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field} not found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields for selected record set.

We plot histogram and boxplot for the numeric field and a bar chart for categorical distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if record_sets:
    df = dataframes[record_set_id]
    if numeric_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=5)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        plt.figure(figsize=(6, 3))
        sns.boxplot(df[numeric_field].dropna())
        plt.title(f"Boxplot of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(7, 4))
        sns.countplot(x=group_field, data=df)
        plt.title(f"Value Counts of {group_field}")
        plt.xlabel(group_field)
        plt.ylabel('Count')
        plt.show()

## 6. Conclusion
We loaded the FAIR^2 dataset using `mlcroissant`, reviewed metadata, extracted all record sets using their unique `@id`s, and performed initial EDA including filtering, normalization, grouping, and visualizations with all references to dataset entities made by their `@id`.

Due to the small, case-based nature of the dataset, the data is best suited for clinical illustration and rare disease research rather than robust predictive modeling. Always use `@id` for reliable entity referencing in Croissant datasets!
